# LibreFOMO — SJSU Headcount Training & Visualization

End-to-end notebook that:
1. Clones [bencejdanko/libreyolo](https://github.com/bencejdanko/libreyolo) and installs it
2. Downloads the **SJSU Headcount Scene-1** dataset from HuggingFace (`bdanko/sjsu-headcount-scene-1`)
3. Trains **LibreFOMO-s** (the lightweight MobileNetV2 point-localization model) for 10 epochs
4. Visualizes the predictions on validation images — predicted person locations as coloured circles overlaid on the original image

Run on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

## 1 — Environment Setup

In [ ]:
import subprocess, sys

# Clone the library (skip if already present)
import os
if not os.path.exists("libreyolo"):
    subprocess.run(
        ["git", "clone", "https://github.com/bencejdanko/libreyolo"],
        check=True
    )
    print("Cloned libreyolo")
else:
    print("libreyolo already present — skipping clone")

In [ ]:
# Install the library and training dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "libreyolo", "scipy"],
    check=True
)
print("Installation complete")

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

## 2 — Download Dataset

`bdanko/sjsu-headcount-scene-1` is a YOLO-format dataset (standard `data.yaml` + `train/images/` + `valid/images/` layout).
It is cloned via `git` the same way all other LibreYOLO training datasets are fetched (marbles, RF5, etc.).

In [ ]:
from pathlib import Path
import yaml
from huggingface_hub import snapshot_download

DATASET_ROOT = Path("sjsu-headcount-scene-1")
HF_REPO = "bdanko/sjsu-headcount-scene-1"

if not DATASET_ROOT.exists() or not (DATASET_ROOT / "data.yaml").exists():
    print(f"Downloading dataset {HF_REPO} via HF Hub...")
    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        local_dir=str(DATASET_ROOT),
        local_dir_use_symlinks=False
    )
    print("Dataset downloaded")
else:
    print("Dataset already present — skipping download")

# Patch data.yaml: replace relative path '.' with the absolute cloned directory
data_yaml_path = DATASET_ROOT / "data.yaml"
data_cfg = yaml.safe_load(data_yaml_path.read_text())
data_cfg["path"] = str(DATASET_ROOT.resolve())
data_yaml_path.write_text(yaml.dump(data_cfg, default_flow_style=False))

print(f"\nDataset config:")
print(f"  path : {data_cfg['path']}")
print(f"  train: {data_cfg['train']}")
print(f"  val  : {data_cfg.get('val', data_cfg.get('valid', 'N/A'))}")
print(f"  nc   : {data_cfg['nc']} ({data_cfg['names']})")

In [ ]:
# Quick sanity check — count images and labels
def count_split(split_dir):
    imgs = list(split_dir.rglob("*.jpg")) + list(split_dir.rglob("*.png"))
    # Try split_dir.parent / "labels" first (e.g., train/labels)
    lbl_dir = split_dir.parent / "labels"
    if not lbl_dir.exists():
        # Fallback to standard Roboflow layout (labels/train)
        lbl_dir = split_dir.parent.parent / "labels" / split_dir.name
    lbls = list(lbl_dir.rglob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

train_imgs, train_lbls = count_split(DATASET_ROOT / "train" / "images")
val_imgs, val_lbls     = count_split(DATASET_ROOT / "valid" / "images")
print(f"Train : {train_imgs} images, {train_lbls} label files")
print(f"Val   : {val_imgs} images, {val_lbls} label files")

## 3 — Train LibreFOMO-s

LibreFOMO-s uses a **MobileNetV2 backbone (α=0.35)** with a single-pixel detection head.
Input resolution is **96×96** → output grid **12×12** (8× downsample).

Training uses:
- Adam, lr=3e-4, ReduceLROnPlateau on val F1
- Weighted CrossEntropy (background=1×, foreground=100×)
- No augmentation (plain resize + ImageNet normalisation)
- Val sweep over confidence thresholds every epoch

In [ ]:
from libreyolo.models.librefomo.model import LibreFOMO

EPOCHS     = 10
BATCH      = 16
MODEL_SIZE = "s"   # "s" | "m" | "l"
PROJECT    = "runs/librefomo"
RUN_NAME   = f"sjsu_headcount_{MODEL_SIZE}"

model = LibreFOMO(model_path=None, size=MODEL_SIZE, nb_classes=1, device=device)
print(f"Model: LibreFOMO-{MODEL_SIZE}")
print(f"  Input size : {model.input_size}×{model.input_size}")
print(f"  Grid size  : {model.input_size//8}×{model.input_size//8}")
print(f"  Classes    : {model.nc}")

In [ ]:
results = model.train(
    allow_experimental=True,
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    lr0=3e-4,
    eval_interval=1,
    workers=2,
    device=device,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience=0,
)

save_dir = Path(results["save_dir"])
print(f"\nTraining complete. Saved to: {save_dir}")

## 4 — Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

epoch_losses = results.get("epoch_losses", [])
val_metrics  = [m for m in results.get("val_metrics", []) if m]

f1s    = [m.get("metrics/F1", 0)         for m in val_metrics]
precs  = [m.get("metrics/precision", 0)  for m in val_metrics]
recs   = [m.get("metrics/recall", 0)     for m in val_metrics]
epochs = range(1, len(epoch_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"LibreFOMO-{MODEL_SIZE} — SJSU Headcount Scene 1", fontsize=13, fontweight="bold")

# Loss
ax = axes[0]
ax.plot(epochs, epoch_losses, color="#e05c4b", linewidth=2, marker="o", markersize=5)
ax.set_title("Training Loss", fontsize=11)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)

# F1 / Precision / Recall
ax = axes[1]
ep_val = range(1, len(f1s) + 1)
ax.plot(ep_val, f1s,   color="#3b82f6", linewidth=2, marker="o", markersize=5, label="F1")
ax.plot(ep_val, precs, color="#10b981", linewidth=2, marker="s", markersize=4, label="Precision", linestyle="--")
ax.plot(ep_val, recs,  color="#f59e0b", linewidth=2, marker="^", markersize=4, label="Recall",    linestyle="--")
ax.set_title("Validation Metrics", fontsize=11)
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1.05)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(save_dir / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

if val_metrics:
    last = val_metrics[-1]
    print(f"\nFinal epoch metrics:")
    print(f"  F1        : {last.get('metrics/F1', 0):.4f}")
    print(f"  Precision : {last.get('metrics/precision', 0):.4f}")
    print(f"  Recall    : {last.get('metrics/recall', 0):.4f}")
    print(f"  MeanDist  : {last.get('metrics/mean_distance', 0):.3f} grid cells")

## 5 — Load Best Checkpoint

In [ ]:
from libreyolo import LibreYOLO

weights_dir = save_dir / "weights"
best_pt  = weights_dir / "best.pt"
last_pt  = weights_dir / "last.pt"
ckpt_path = best_pt if best_pt.exists() else last_pt

trained = LibreYOLO(str(ckpt_path), device=device)
print(f"Loaded: {ckpt_path.name}")
print(f"  family : {trained.family}")
print(f"  size   : {trained.size}")
print(f"  imgsz  : {trained.input_size}")
print(f"  nc     : {trained.nc}")

## 6 — Inference Visualisation

For each validation image we:
- Run `model.predict()` → `result.points` (x, y in **grid** coordinates, scaled back to image pixels)
- Draw **cyan circles** for predictions
- Draw **green crosses** for ground-truth centers (from YOLO label files)
- Show the count predicted vs. count ground-truth

In [ ]:
import numpy as np
from PIL import Image
import random

# ── helpers ──────────────────────────────────────────────────────────────────

def load_gt_points(label_path: Path, img_w: int, img_h: int):
    """Read YOLO-format label file and return list of (px, py) in image pixels."""
    if not label_path.exists():
        return []
    pts = []
    for line in label_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cx, cy = float(parts[1]), float(parts[2])
        pts.append((cx * img_w, cy * img_h))
    return pts


def draw_predictions(img: Image.Image, result, gt_points, conf_thresh=0.25):
    """
    Return an annotated PIL image.
    - Cyan filled circle  = predicted person location
    - Green cross (+)     = ground-truth person center
    """
    import PIL.ImageDraw as ImageDraw
    import PIL.ImageFont as ImageFont

    W, H = img.size
    canvas = img.copy().convert("RGB")
    draw   = ImageDraw.Draw(canvas)

    # Predicted points — result.points.xy are in grid-cell coordinates;
    # scale back to image pixels (grid = imgsz // 8)
    grid_size  = trained.input_size // 8
    cell_w     = W / grid_size
    cell_h     = H / grid_size
    RADIUS     = max(4, int(min(W, H) * 0.018))

    n_pred = 0
    if result.points is not None:
        pts   = result.points
        xy    = pts.xy.cpu().numpy()    # (N, 2) in grid coords
        confs = pts.conf.cpu().numpy()  # (N,)
        for (gx, gy), c in zip(xy, confs):
            if c < conf_thresh:
                continue
            px = gx * cell_w + cell_w / 2
            py = gy * cell_h + cell_h / 2
            draw.ellipse(
                [(px - RADIUS, py - RADIUS), (px + RADIUS, py + RADIUS)],
                fill=(0, 220, 255),
                outline=(0, 150, 200),
                width=2,
            )
            n_pred += 1

    # Ground-truth
    ARM = max(3, RADIUS)
    for (gx, gy) in gt_points:
        draw.line([(gx - ARM, gy), (gx + ARM, gy)], fill=(0, 230, 80), width=3)
        draw.line([(gx, gy - ARM), (gx, gy + ARM)], fill=(0, 230, 80), width=3)

    # Legend text
    label = f"Pred: {n_pred}  GT: {len(gt_points)}"
    margin = 6
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 18)
    except Exception:
        font = ImageFont.load_default()
    bbox = draw.textbbox((margin, margin), label, font=font)
    draw.rectangle([bbox[0]-3, bbox[1]-3, bbox[2]+3, bbox[3]+3], fill=(0, 0, 0, 180))
    draw.text((margin, margin), label, fill=(255, 255, 255), font=font)

    return canvas, n_pred


print("Helpers defined")

In [ ]:
# Collect val images
val_dir   = DATASET_ROOT / "valid" / "images"
label_dir = val_dir.parent / "labels"
if not label_dir.exists():
    label_dir = DATASET_ROOT / "labels" / "valid"

val_images = sorted(val_dir.glob("*.jpg")) + sorted(val_dir.glob("*.png"))
print(f"Found {len(val_images)} validation images")

# Pick a random sample to visualise (up to 12)
N_SHOW = min(12, len(val_images))
sample  = random.sample(val_images, N_SHOW)
print(f"Visualising {N_SHOW} images")

In [ ]:
CONF_THRESHOLD = 0.25

annotated = []
total_pred = total_gt = 0

for img_path in sample:
    pil_img = Image.open(img_path).convert("RGB")
    W, H    = pil_img.size

    label_path = label_dir / (img_path.stem + ".txt")
    gt_pts     = load_gt_points(label_path, W, H)

    result = trained.predict(pil_img, conf=CONF_THRESHOLD, max_det=300)

    canvas, n_pred = draw_predictions(pil_img, result, gt_pts, conf_thresh=CONF_THRESHOLD)
    annotated.append(canvas)
    total_pred += n_pred
    total_gt   += len(gt_pts)

print(f"Predictions: {total_pred} total  |  Ground truth: {total_gt} total")

In [ ]:
# Display grid: up to 4 columns
N_COLS = min(4, N_SHOW)
N_ROWS = (N_SHOW + N_COLS - 1) // N_COLS

THUMB_W, THUMB_H = 320, 240

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 4.2, N_ROWS * 3.2))
fig.suptitle(
    f"LibreFOMO-{MODEL_SIZE} · SJSU Headcount Scene 1 · conf≥{CONF_THRESHOLD}\n"
    f"● cyan = predicted  + green = ground truth",
    fontsize=12, fontweight="bold", y=1.01
)

for i, (ax, img) in enumerate(zip(np.array(axes).flat, annotated)):
    thumb = img.resize((THUMB_W, THUMB_H), Image.BILINEAR)
    ax.imshow(thumb)
    ax.axis("off")

# Hide unused axes
for ax in np.array(axes).flat[len(annotated):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(save_dir / "val_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_dir / 'val_predictions.png'}")

## 7 — Count Accuracy Summary

For each validation image, compare **predicted count** vs **ground-truth count** and compute mean absolute error (MAE).

In [ ]:
all_images = sorted(val_dir.glob("*.jpg")) + sorted(val_dir.glob("*.png"))

pred_counts = []
gt_counts   = []

trained.nn.eval()
with torch.no_grad():
    for img_path in all_images:
        pil_img    = Image.open(img_path).convert("RGB")
        W, H       = pil_img.size
        label_path = label_dir / (img_path.stem + ".txt")
        gt_pts     = load_gt_points(label_path, W, H)
        result     = trained.predict(pil_img, conf=CONF_THRESHOLD, max_det=300)
        n_pred     = len(result) if result.points is not None else 0
        pred_counts.append(n_pred)
        gt_counts.append(len(gt_pts))

pred_counts = np.array(pred_counts)
gt_counts   = np.array(gt_counts)
mae         = np.mean(np.abs(pred_counts - gt_counts))
corr        = np.corrcoef(pred_counts, gt_counts)[0, 1]

print(f"Validation set ({len(all_images)} images)")
print(f"  MAE (count error) : {mae:.2f} persons")
print(f"  Pearson r          : {corr:.4f}")
print(f"  Mean GT count      : {gt_counts.mean():.1f}")
print(f"  Mean pred count    : {pred_counts.mean():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"LibreFOMO-{MODEL_SIZE} · Count Accuracy on Validation Set",
             fontsize=12, fontweight="bold")

# Scatter: pred vs GT
ax = axes[0]
ax.scatter(gt_counts, pred_counts, alpha=0.55, color="#3b82f6", edgecolors="white",
           linewidths=0.5, s=50)
max_val = max(gt_counts.max(), pred_counts.max()) + 1
ax.plot([0, max_val], [0, max_val], "--", color="#94a3b8", linewidth=1.5, label="Perfect")
ax.set_xlabel("Ground-truth count")
ax.set_ylabel("Predicted count")
ax.set_title(f"Pred vs GT  (r={corr:.3f})")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Histogram of errors
ax = axes[1]
errors = pred_counts - gt_counts
bins   = np.arange(errors.min() - 0.5, errors.max() + 1.5, 1)
ax.hist(errors, bins=bins, color="#6366f1", edgecolor="white", linewidth=0.6)
ax.axvline(0, color="#ef4444", linewidth=1.8, linestyle="--")
ax.set_xlabel("Count error (pred − GT)")
ax.set_ylabel("Images")
ax.set_title(f"Error distribution  (MAE={mae:.2f})")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(save_dir / "count_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_dir / 'count_accuracy.png'}")

---
## Summary

| | |
|---|---|
| **Model** | `LibreFOMO-s` (MobileNetV2 α=0.35, 96×96) |
| **Dataset** | `bdanko/sjsu-headcount-scene-1` (YOLO format) |
| **Training** | Adam lr=3e-4, ReduceLROnPlateau on F1, 100× fg weight |
| **Inference** | `result.points.xy` → grid coords → image pixels |
| **Evaluation** | Threshold sweep over conf × NMS-radius → best F1 |

The checkpoint (`best.pt`) saved in `runs/librefomo/` is compatible with the LibreYOLO factory:
```python
from libreyolo import LibreYOLO
model = LibreYOLO("runs/librefomo/sjsu_headcount_s/weights/best.pt")
result = model.predict(image)
print(result.points.xy)   # person locations in grid coords
```